# ML Challenge 2026: Business Entity Resolution
## Notebook 01: Exploratory Data Analysis & Integrity Audit

This notebook performs the comprehensive structural audit of the challenge dataset:
- Train & Test source shapes and schemas
- Null/missingness analysis
- Country distribution (detecting unseen test countries such as France)
- Ground truth match distribution, multiplicity, and singleton (0-match) analysis
- Identifier integrity, prefix checks, and reference resolution.

In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Load precomputed comprehensive EDA audit
report_path = Path('../experiments/eda_report.json')
with open(report_path, 'r', encoding='utf-8') as f:
    eda = json.load(f)

print('EDA Audit Loaded Successfully.')

### 1. Dataset Dimensions & Summary

In [2]:
summary_rows = []
for split in ['train_source1', 'train_source2', 'train_source3', 'test_source1', 'test_source2', 'test_source3']:
    d = eda[split]
    summary_rows.append({
        'File': d['filename'],
        'Rows': f"{d['total_rows']:,}",
        'Unique IDs': f"{d['unique_entity_ids']:,}",
        'Duplicate IDs': d['duplicate_entity_ids'],
        'Missing Name': d['missing_or_empty_name'],
        'Missing Address': f"{d['missing_or_empty_addr']:,}",
        'Countries': str(d['country_distribution'])
    })
df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

### 2. Ground Truth Analysis & Singleton Distribution

In [3]:
gt = eda['ground_truth']
print(f"Total Source 1 in GT: {gt['total_rows']:,}")
print(f"Singletons (0 matches): {gt['singleton_count (0 matches)']:,} ({gt['singleton_percentage']}%)")
print(f"Mean matches: {gt['avg_matches']}, Median: {gt['median_matches']}, Max: {gt['max_matches']}")
print(f"Total True Links: Source 2 = {gt['total_s2_links']:,}, Source 3 = {gt['total_s3_links']:,}")

freq_df = pd.DataFrame(list(gt['match_count_frequency'].items()), columns=['Matches_per_S1', 'Count'])
freq_df['Matches_per_S1'] = freq_df['Matches_per_S1'].astype(int)
freq_df.sort_values('Matches_per_S1', inplace=True)
print(freq_df.to_string(index=False))

### 3. Country Analysis: Train vs Test

In [4]:
train_c = eda['train_source1']['country_distribution']
test_c = eda['test_source1']['country_distribution']
c_df = pd.DataFrame({'Train_S1': train_c, 'Test_S1': test_c}).fillna(0).astype(int)
print(c_df)